<p style="padding: 10px; border: 1px solid black;">
<img src="../mlu_utils/images/MLU-NEW-logo.png" alt="drawing" width="400"/> <br/>
<div style="background-color: #1A5276; padding: 20px; border-radius: 10px; text-align: center; margin-bottom: 30px;">
    <h1 style="color: white; margin: 0;">MLU: Responsible AI</h1>
    <h2 style="color: white; margin-top: 15px;">Lab 4b: Watermarking</h2>
</div>

<!-- Compact Lab Introduction with Activity/Challenge Explanation -->
<div style="background-color: #F8F9F9; padding: 15px; border-radius: 5px; margin: 20px 0;">
    <h4 style="color: #2E4053; margin-top: 0;">About This Lab</h4>
    <p>Throughout this lab, you will encounter two types of interactive elements:</p>
    <table style="width: 100%; border-collapse: collapse; margin: 15px 0;">
        <tr>
            <td style="text-align: center; padding: 10px; width: 50%;">
                <img src="../mlu_utils/images/activity.png" alt="Activity" width="125"/>
            </td>
            <td style="text-align: center; padding: 10px; width: 50%;">
                <img src="../mlu_utils/images/challenge.png" alt="Challenge" width="125"/>
            </td>
        </tr>
        <tr>
            <td style="text-align: center; padding: 10px; background-color: #EBF5FB;">
                <p>No coding is needed for an activity. You try to understand a concept, <br/>answer questions, or run a code cell.</p>
            </td>
            <td style="text-align: center; padding: 10px; background-color: #FEF9E7;">
                <p>Challenges are where you test your understanding by implementing something new or taking a short quiz.</p>
            </td>
        </tr>
    </table>
    <p>Please work through this notebook from top to bottom to avoid errors due to missing code or context.</p>
</div>

<!-- Table of Contents with All Section Levels -->
<div style="background-color: #EBF5FB; padding: 15px; border-radius: 5px; margin-bottom: 30px;">
    <h2 style="color: #2874A6; border-bottom: 1px solid #2874A6; padding-bottom: 5px;">Table of Contents</h2>
    <p><a href="#section1" style="color: #2E86C1; font-weight: bold; text-decoration: none;">1. Install and import libraries</a></p>
    <p><a href="#section2" style="color: #2E86C1; font-weight: bold; text-decoration: none;">2. Watermarking for authentication</a></p>
    <p><a href="#section3" style="color: #2E86C1; font-weight: bold; text-decoration: none;">3. Quizzes</a></p>
</div>

---

This notebook demonstrates how to use various techniques that can help improve the safety and security of LLM-backed applications. The coding examples cover watermarking as an authentication technique.

<!-- Section Header -->
<div id="section1" style="border-left: 5px solid #3498DB; padding-left: 15px; margin: 40px 0 20px 0;">
    <h2 style="color: #2874A6;">1. Install and import libraries</h2>
</div>

Let's start by installing all required packages as specified in the `requirements.txt` file and importing several libraries.

In [ ]:
%%capture
!pip3 install -r ../requirements.txt --quiet
!rm -rf lm-watermarking
!git clone https://github.com/jwkirchenbauer/lm-watermarking.git --quiet

In [ ]:
import warnings, sys, os

warnings.filterwarnings("ignore")
cwd = os.getcwd()
print(f"current working directory >>>>> {cwd} \n")
sys.path.append(cwd + "/lm-watermarking/")

import json
from IPython.display import Markdown

<!-- Section Header -->
<div id="section2" style="border-left: 5px solid #3498DB; padding-left: 15px; margin: 40px 0 20px 0;">
    <h2 style="color: #2874A6;">2. Watermarking for authentication</h2>
</div>

Potential harms of LLMs can be mitigated by watermarking model output, i.e., **embedding signals into generated text that are invisible to humans but algorithmically detectable from a short span of tokens**. Watermarks can be embedded with negligible impact on text quality, and can be detected using efficient open-source algorithms without access to the language model API or model parameters. The watermark works by selecting a randomized set of "green" tokens before a word is generated, and then softly promoting use of green tokens during sampling. For more details about watermarks for LLMs have a look at the paper [A Watermark for Large Language Models](https://arxiv.org/abs/2301.10226).

First, you need to load in a tokenizer and model that allows access to the tokens and associated logit values. This means, you will need to use a Huggingface 🤗 or other third-party LLM that you can run locally. Bedrock-hosted models can be queried but not downloaded, thus they are not apt for this demo.

The following uses a tiny LLM, [`dlite-v2-124m`](https://huggingface.co/aisquared/dlite-v2-124m), derived from OpenAI's smallest GPT-2 model and fine-tuned on a single GPU. `dlite-v2-124m` is **not a state-of-the-art model**. We are using it here to demonstrate watermarking in a lean setup with a CPU instance. If you have access to larger, GPU-enabled instances, feel free to try larger models available in Huggingface. 

In [ ]:
import IPython

# you can uncomment and auto-restart if you run into issues 
#IPython.get_ipython().kernel.do_shutdown(restart=True)

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch 

model_id = "aisquared/dlite-v2-124m"

tokenizer = AutoTokenizer.from_pretrained(
    model_id,
    padding_side="left",
    device_map='auto'
)
tokenizer.eos_token_id  = tokenizer.pad_token_id

# Load tiny model in BF16 precision
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    device_map="auto",
    torch_dtype=torch.bfloat16,
)

With the model and tokenizer you can now generate output tokens and pass the logits values to the watermark processor that will add certain random tokens. `WatermarkLogitsProcessor` loads a 🤗 language model that can perform text generation via `model.generate`, and prepares to call the generation method with a special LogitsProcessor that implements watermarking at the current hyperparameter values. The most important parameters to specify are:
- `gamma`: Gamma denotes the fraction of the vocabulary that will be in each green list. 
- `delta`: The magnitude of the logit bias delta determines the strength of the watermark. 

As a baseline generation setting, default values of `gamma=0.25` and `delta=2.0` are suggested. Reduce `delta` if text quality is negatively impacted. 

In [ ]:
from extended_watermark_processor import WatermarkLogitsProcessor
from transformers import LogitsProcessorList

# instantiate watermarking processor
watermark_processor = WatermarkLogitsProcessor(
    vocab=list(tokenizer.get_vocab().values()),
    gamma=0.25,
    delta=2.0,
    seeding_scheme="selfhash",
)

# tokenize input
tokenized_input = tokenizer("What did you do today?", return_tensors="pt").to(model.device)

# generate output tokens and parse through watermarking
output_tokens = model.generate(
    **tokenized_input,
    pad_token_id=50256,
    logits_processor=LogitsProcessorList([watermark_processor])
)

# isolate newly generated tokens as only those are watermarked, the input/prompt is not
output_tokens = output_tokens[:, tokenized_input["input_ids"].shape[-1] :]

# convert back to text
output_text = tokenizer.batch_decode(output_tokens, skip_special_tokens=True)[0]

Have a look at the resulting text.

In [ ]:
Markdown(output_text)

Let's now try to detect the watermarked text. 

The `WatermarkDetector` is the detector for all watermarks imprinted with `WatermarkLogitsProcessor`. It needs to be given the exact same settings that were given during text generation  to replicate the watermark greenlist generation and so detect the watermark. This includes the correct device that was used during text generation, the correct tokenizer, the correct seeding_scheme name, and parameters.

The detector below shows a high confidence that the input text has been watermarked.  


In [ ]:
from extended_watermark_processor import WatermarkDetector

watermark_detector = WatermarkDetector(
    vocab=list(tokenizer.get_vocab().values()),
    gamma=0.25,  # should match original setting
    seeding_scheme="selfhash",  # should match original setting
    device=model.device,  # must match the original rng device type
    tokenizer=tokenizer,
    z_threshold=4.0,
    normalizers=[],
    ignore_repeated_ngrams=True,
)

score_dict = watermark_detector.detect(
    output_text
)  # or any other text of interest to analyze

score_dict

Now compare with the watermarker detector acting on regularly generated text. In this case, the detector correctly predicts that the text is not watermarked. 

In [ ]:
# tokenize input
tokenized_input = tokenizer("What did you do today?", return_tensors="pt").to(model.device)

# generate output tokens and parse through watermarking
output_tokens = model.generate(
    **tokenized_input,
    pad_token_id=50256,
)

# isolate newly generated tokens as only those are watermarked, the input/prompt is not
output_tokens = output_tokens[:, tokenized_input["input_ids"].shape[-1] :]

# convert back to text
output_text = tokenizer.batch_decode(output_tokens, skip_special_tokens=True)[0]

score_dict = watermark_detector.detect(output_text)

score_dict

<!-- Activity Box -->
<div style="background-color: #EBF5FB; border-left: 5px solid #3498DB; padding: 15px; border-radius: 5px; margin: 20px 0; display: flex; align-items: flex-start;">
    <div style="flex: 0 0 60px; margin-right: 15px;">
        <img src="../mlu_utils/images/activity.png" alt="Activity" width="200" style="max-width: 100%; height: auto;">
    </div>
    <div style="flex: 1;">
        <h4 style="color: #2874A6; margin-top: 0;">Activity: Try your own watermarking</h4>
        <p>Try your own prompt and add a watermark authentication to it. Also try to change the different parameters for <code>WatermarkLogitsProcessor</code> to see how the output is changing.</p>
        <p><strong>Note:</strong> due to the limited capabilities of the <code>dlite-v2-124m</code> model, not all experiments might work. You can try more capable LLMs if you have access to larger instances to run them.</p>
    </div>
</div>

In [ ]:
############## CODE HERE ####################


############## END OF CODE ##################

<!-- Section Header -->
<div id="section3" style="border-left: 5px solid #3498DB; padding-left: 15px; margin: 40px 0 20px 0;">
    <h2 style="color: #2874A6;">3. Quizzes</h2>
    <p>Well done on completing the lab! Now, it's time for a brief knowledge assessment.</p>
</div>

<!-- Challenge Box -->
<div style="background-color: #FEF9E7; border-left: 5px solid #F1C40F; padding: 15px; border-radius: 5px; margin: 20px 0; display: flex; align-items: flex-start;">
    <div style="flex: 0 0 60px; margin-right: 15px;">
        <img src="../mlu_utils/images/challenge.png" alt="Challenge" width="200" style="max-width: 100%; height: auto;">
    </div>
    <div style="flex: 1;">
        <h4 style="color: #B7950B; margin-top: 0;">Challenge: Knowledge Assessment</h4>
        <p>Answer the following questions to test your understanding of embeddings, document loaders and RAG workflows.</p>
    </div>
</div>

In [ ]:
import sys
sys.path.append('..')

from mlu_utils.quiz_questions import lab4b_question1

lab4b_question1.display()

In [ ]:
# run this cell when you finish the notebook to clean up your environment
!rm -rf ./lm-watermarking

<div style="background-color: #EBF5FB; padding: 15px; border-radius: 5px; margin: 30px 0;">
    <h3 style="color: #2874A6; border-bottom: 1px solid #85C1E9; padding-bottom: 5px;">Conclusion</h3>
    <p>In this lab, you have:</p>
    <ul>
        <li>Learned about watermarking as an authentication technique for LLM outputs</li>
        <li>Implemented watermarking using the WatermarkLogitsProcessor</li>
        <li>Detected watermarks in generated text using the WatermarkDetector</li>
        <li>Explored how watermarking can help monitor and audit LLM usage</li>
    </ul>
    <h4 style="color: #2874A6; margin-top: 15px;">Additional Resources</h4>
    <ul>
        <li><a href="https://github.com/microsoft/promptbench">Microsoft PromptBench</a></li>
        <li><a href="https://www.promptingguide.ai/techniques">Prompting Guide Techniques</a></li>
        <li><a href="https://github.com/uptrain-ai/uptrain">UpTrain AI</a></li>
    </ul>
</div>

<p style="padding: 10px; border: 1px solid black;">
<img src="../mlu_utils/images/MLU-NEW-logo.png" alt="drawing" width="400"/> <br/>

# Thank you!